# 02 – Regex baseline

A hand-written regex that finds Gleason pattern pairs in the report text. It is the
baseline the two LLMs are compared against, and it uses the same parser, metrics and
run logging (`common.py`).

The regex looks for two kinds of candidates:

1. **Pairs** written as `X + Y`, e.g. `Gleason score 3 + 4 = 7` or `GLEASON SCORE 7 (4+3)`,
   with a Gleason-related word shortly before.
2. **Labelled patterns**, e.g. `Primary pattern: 3/5 ... Secondary pattern: 4/5` or
   `PRIMARY GLEASON GRADE: 4. SECONDARY GLEASON GRADE: 3`.

When a report contains several different pairs, `multi_score_rule` decides which one is the
answer (the same rule must be stated in the LLM prompt). If nothing is found, the answer is
"not found" (`null`, `null`).

The output for each report is a JSON string in the same schema as the LLM answers, plus a
`matches` list with every candidate found, so the choice can be checked afterwards.

Runs on CPU in seconds, so it is fine on the login node.

In [ ]:
import json
import re
import sys

import pandas as pd

sys.path.insert(0, ".")  # common.py lives next to this notebook
import common

In [ ]:
config = {
    "method": "regex",
    "model_id": None,
    "subset": "tuning",  # "tuning" or "eval"
    "n_reports": None,  # None = all reports in the subset; a number for a quick test
    # Which pair is the answer when a report has several different ones:
    #   "highest": highest score, ties broken by the higher primary pattern
    #   "first":   the first one in the report
    "multi_score_rule": "highest",
    # 1. "X + Y", optionally followed by "= Z". X and Y are not part of a longer chain like 3+4+5.
    "pair_regex": r"(?<![\d+])([1-5])(?!\d)\s*\+\s*([1-5])(?!\d)(?!\s*\+\s*\d)(?:\s*=\W{0,3}(\d{1,2})(?!\d))?",
    # A pair only counts if one of these words appears this many characters before it
    "pair_context_regex": r"gleason|grade|pattern|score|sum",
    "pair_context_chars": 60,
    # 2. "primary ... pattern/grade ... X ... secondary ... pattern/grade ... Y".
    #    Letters of "pattern" and "grade" may be split by OCR spaces ("Pat t er n", "Gr ade").
    "labelled_regex": (
        r"primary\W{0,3}(?:gleason\W{0,3})?(?:p\s?a\s?t\s?t\s?e\s?r\s?n|g\s?r\s?a\s?d\s?e)\W{0,5}"
        r"(?:g\s?r\s?a\s?d\s?e\W{0,3})?([1-5])(?!\d)"
        r".{0,150}?"
        r"secondary\W{0,3}(?:gleason\W{0,3})?(?:p\s?a\s?t\s?t\s?e\s?r\s?n|g\s?r\s?a\s?d\s?e)\W{0,5}"
        r"(?:g\s?r\s?a\s?d\s?e\W{0,3})?([1-5])(?!\d)"
    ),
    "log_run": False,
    "note": "",
}

## Extraction

In [ ]:
def find_candidates(text, config):
    """Find every Gleason pattern pair in a report.

    Args:
        text: the report text.
        config: the config dict (regexes and context settings).

    Returns:
        List of dicts with primary, secondary, start (character position) and match (the
        matched text), sorted by position in the report.
    """
    candidates = []
    flags = re.IGNORECASE | re.DOTALL

    for m in re.finditer(config["pair_regex"], text, flags):
        primary, secondary = int(m.group(1)), int(m.group(2))
        # If the sum is written ("3 + 4 = 7"), it must agree; otherwise it is probably not a Gleason pair
        if m.group(3) is not None and int(m.group(3)) != primary + secondary:
            continue
        before = text[max(0, m.start() - config["pair_context_chars"]):m.start()]
        if not re.search(config["pair_context_regex"], before, flags):
            continue
        candidates.append({"primary": primary, "secondary": secondary,
                           "start": m.start(), "match": m.group(0)})

    for m in re.finditer(config["labelled_regex"], text, flags):
        candidates.append({"primary": int(m.group(1)), "secondary": int(m.group(2)),
                           "start": m.start(), "match": m.group(0)})

    return sorted(candidates, key=lambda c: c["start"])


def choose_candidate(candidates, rule):
    """Pick the answer from the candidates, following the multi-score rule.

    Args:
        candidates: list from find_candidates (may be empty).
        rule: "highest" (highest sum, then highest primary) or "first" (earliest in the report).

    Returns:
        The chosen candidate dict, or None if there are no candidates.
    """
    if not candidates:
        return None
    if rule == "highest":
        return max(candidates, key=lambda c: (c["primary"] + c["secondary"], c["primary"]))
    if rule == "first":
        return candidates[0]
    raise ValueError(f"Unknown multi_score_rule: {rule}")


def extract(text, config):
    """Run the regex baseline on one report.

    Args:
        text: the report text.
        config: the config dict.

    Returns:
        A JSON string in the answer schema: {"primary", "secondary", "matches"}, where matches
        lists every pair found (as "X+Y") in report order. primary/secondary are null if none.
    """
    candidates = find_candidates(text, config)
    chosen = choose_candidate(candidates, config["multi_score_rule"])
    return json.dumps({
        "primary": chosen["primary"] if chosen else None,
        "secondary": chosen["secondary"] if chosen else None,
        "matches": [f"{c['primary']}+{c['secondary']}" for c in candidates],
    })

## Run

In [ ]:
data = common.load_data(config["subset"], n=config["n_reports"])
raw_outputs = [extract(text, config) for text in data["text"]]

table = common.build_run_table(data, raw_outputs, config)
scores = common.score_run(table)
print(json.dumps({k: v for k, v in scores.items() if k != "confusion_matrix"}, indent=2))
print("\nConfusion matrix (rows: true score, columns: predicted):")
print(pd.DataFrame(scores["confusion_matrix"]).T.fillna(0).astype(int).sort_index(axis=1))

## Errors

Every report where the predicted pair is not the label, with all pairs the regex found.
A label that never appears among the matches is either a regex miss or a report that
does not state the labelled score.

In [ ]:
errors = table[~table["correct_pair"]].copy()
errors["matches"] = [json.loads(raw)["matches"] for raw in errors["raw_output"]]
errors["label"] = errors["label_primary"].astype(str) + "+" + errors["label_secondary"].astype(str)
errors["predicted"] = errors["pred_primary"].astype(str) + "+" + errors["pred_secondary"].astype(str)
print(f"{len(errors)} of {len(table)} pattern pairs wrong")
errors[["patient_id", "status", "label", "predicted", "matches"]]

## "Not found" reports, for checking by hand

Shows every place a Gleason-related word occurs in the reports where the regex found
nothing. Reading these tells whether the report really has no score (counts towards the
ceiling for all methods) or whether the regex missed it.

In [ ]:
texts = data.set_index("patient_id")["text"]
for patient_id in table.loc[table["status"] == "not_found", "patient_id"]:
    text = texts[patient_id]
    print(f"===== {patient_id}")
    for m in re.finditer(r"gl\s?e?\s?a\s?s\s?o\s?n|primary|secondary", text, re.IGNORECASE):
        print("   ...", text[max(0, m.start() - 40):m.start() + 120].replace("\n", " "))

## Save

In [ ]:
common.log_run(config, table, scores)